## Idea:
* Use month trend from [Only Trends](https://www.kaggle.com/code/vitalykudelya/only-trends)
* Divide NPX values of a protein into several groups and find the best shift after month trend predicitons for each group
* Sum predictions from the month trend and the corresponding NPX group shift

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

import plotly.express as px

import amp_pd_peptide

from scipy.optimize import minimize

In [ ]:
def smape_plus_1(y_true, y_pred):
    y_true_plus_1 = y_true + 1
    y_pred_plus_1 = y_pred + 1
    metric = np.zeros(len(y_true_plus_1))
    
    numerator = np.abs(y_true_plus_1 - y_pred_plus_1)
    denominator = ((np.abs(y_true_plus_1) + np.abs(y_pred_plus_1)) / 2)
    
    mask_not_zeros = (y_true_plus_1 != 0) | (y_pred_plus_1 != 0)
    metric[mask_not_zeros] = numerator[mask_not_zeros] / denominator[mask_not_zeros]
    
    return 100 * np.nanmean(metric)

## Generate Train Dataset

In [ ]:
train_clinical_all = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/train_clinical_data.csv')
proteins = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/train_proteins.csv')

proteins_features = pd.pivot_table(proteins, values='NPX', index='visit_id', columns='UniProt', aggfunc='sum')

train_clinical_all = train_clinical_all.merge(
    proteins_features,
    left_on='visit_id',
    right_index=True,
    how='left'
)

# # Define the integrated moving mean function
# def integrated_moving_sum(values):
#     return values.expanding().sum()

# # Create the pivot table using the integrated moving mean
# proteins_features = pd.pivot_table(proteins, 
#                                    values='NPX', 
#                                    index='visit_id', 
#                                    columns='UniProt', 
#                                    aggfunc=integrated_moving_sum)


# # # Calculate the exponentially weighted sum for each column
# # exponential_weights = proteins_features.ewm(alpha=0.9).mean()

# # # Merge the exponentially weighted sum with the train_clinical_all DataFrame
# train_clinical_all = train_clinical_all.merge(
#     proteins_features,
#     left_on='visit_id',
#     right_index=True,
#     how='left'
# )


In [ ]:


#import numpy as np

# # Forward-fill (ffill)
# train_clinical_all[proteins_features.columns] = train_clinical_all.groupby('patient_id')[proteins_features.columns].\
#                                                 fillna(method='ffill')

# # Backward-fill (bfill)
# train_clinical_all[proteins_features.columns] = train_clinical_all.groupby('patient_id')[proteins_features.columns].\
#                                                 fillna(method='bfill')
import warnings
warnings.filterwarnings('ignore')

# Mean imputation
train_clinical_all[proteins_features.columns] = train_clinical_all.groupby('patient_id')[proteins_features.columns].\
                                               transform(lambda x: x.fillna(x.median()))

# train_clinical_all[proteins_features.columns] = np.mean([train_clinical_all[proteins_features.columns].ffill(),
#                                                          train_clinical_all[proteins_features.columns].bfill(),
#                                                          train_clinical_all[proteins_features.columns]],
#                                                         axis=0)


In [ ]:
# Calculate the correlation matrix
# corr_matrix = train_clinical_all.corr()
# # Create an empty list to store the non-correlated features
# correlated_features = []

# # Iterate over the columns of the correlation matrix
# for col in corr_matrix.columns:
#     # Check if the absolute correlation value is less than a threshold (e.g., 0.5)
#     if abs(corr_matrix[col]).mean() > 0.31:
#         correlated_features.append(col)
    
#     # Break the loop if we have found the top 3 non-correlated features
#     if len(correlated_features) == 7:
#         break

# # Select the non-correlated features from the grouped data
# print(correlated_features)


In [ ]:
# # Define the target feature
# target_feature = 'updrs_2'

# # Calculate the correlation between each feature and the target
# correlation_with_target = train_clinical_all[correlated_features+[target_feature]].corr()[target_feature]

# # Set a correlation threshold (e.g., 0.7) to determine strong correlation
# correlation_threshold = 0.2

# # Select the features with correlation above the threshold
# strongly_correlated_features = correlation_with_target[correlation_with_target.abs() > correlation_threshold].index.tolist()
# strongly_correlated_features


#['O15240', 'O15394', 'updrs_1']
#['O00533', 'O15240', 'O15394', 'updrs_2']
#['O00533', 'O15240', 'updrs_3']


In [ ]:
train_clinical_all['pred_month'] = train_clinical_all['visit_month']

for plus_month in [6, 12, 24]:
    train_shift = train_clinical_all[['patient_id', 'visit_month', 'pred_month', 'updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']].copy()
    train_shift['visit_month'] -= plus_month
    
    train_shift.rename(columns={f'updrs_{i}': f'updrs_{i}_plus_{plus_month}' for i in range(1, 5)}, inplace=True)
    train_shift.rename(columns={'pred_month': f'pred_month_plus_{plus_month}'}, inplace=True)
    
    train_clinical_all = train_clinical_all.merge(train_shift, how='left', on=['patient_id', 'visit_month'])

train_clinical_all.rename(columns={f'updrs_{i}': f'updrs_{i}_plus_0' for i in range(1, 5)}, inplace=True)
train_clinical_all.rename(columns={'pred_month': f'pred_month_plus_0'}, inplace=True)
train_clinical_all

In [ ]:
# Define a lambda function to fill NaN values with the mode
from scipy.stats import mode
fill_with_mode = lambda x: x.fillna(mode(x).mode[0])


In [ ]:
def calculate_month_trend_predicitons(pred_month, trend):
    if target == 'updrs_4': 
        pred_month = pred_month.clip(60, None)
    return trend[0] + pred_month * trend[1]

target_to_trend = {
    'updrs_1': [5.394793062665313, 0.027091086167821344],
    'updrs_2': [5.469498130092747, 0.02824188329658148],
    'updrs_3': [21.182145576879183, 0.08897763331790556],
    'updrs_4': [-4.434453480103724, 0.07531448585334258]
}

In [ ]:
def calculate_predicitons_protein(pred_month, protein_shift):
    trend_pred_month = target_to_trend[target]
    pred_month_trend = calculate_month_trend_predicitons(pred_month=pred_month, trend=trend_pred_month)
    return np.round(pred_month_trend + protein_shift)

def function_to_minimize(x):
    metric = smape_plus_1(
        y_true=y_true_array, 
        y_pred=calculate_predicitons_protein(
            #protein=protein_array,
            pred_month=pred_month_array,
            protein_shift=x[0]
        )
    )
    return metric

In [ ]:
def find_best_const(train_clinical_all_filtered, target):
    columns_with_target = [f'{target}_plus_{plus_month}' for plus_month in [0, 6, 12, 24]]
    columns_with_pred_month = [f'pred_month_plus_{plus_month}' for plus_month in [0, 6, 12, 24]]
    global y_true_array
    global pred_month_array
    global protein_array
    y_true_array = train_clinical_all_filtered[columns_with_target].values.ravel()
    pred_month_array = train_clinical_all_filtered[columns_with_pred_month].values.ravel()
    #protein_array = np.concatenate([train_clinical_all_filtered[feature].values] * 4)
    result = minimize(
        fun=function_to_minimize,
        x0=[0.0],
        method='Powell'
    ).x[0]
    return result

## Plot shifts

In [ ]:
feature0 = 'O15240'
quantiles = [0, 0.05, 0.95, 1.0]

df_plot = []
for quantile_low, quantile_high in tqdm(zip(quantiles[:-1], quantiles[1:])):
    item = {
        'quantile_low': quantile_low,
        'quantile_high': quantile_high,
        'quantile_middle': (quantile_low + quantile_high) / 2
    }
    quantile_low_value = train_clinical_all[feature0].quantile(quantile_low)
    quantile_high_value = train_clinical_all[feature0].quantile(quantile_high)
    item['quantile_low_value'] = quantile_low_value
    item['quantile_high_value'] = quantile_high_value
    
    if quantile_high == 1:
        quantile_high_value += 0.00001
        
    train_clinical_all_filtered0 = train_clinical_all[
        (train_clinical_all[feature0] >= quantile_low_value)
        & (train_clinical_all[feature0] <= quantile_high_value)
    ]
    for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
        item[f'{target}_shift'] = find_best_const(train_clinical_all_filtered0, target)
    df_plot.append(item)
    
df_plot = pd.DataFrame(df_plot)


In [ ]:
for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
    fig = px.line(
        df_plot,
        y=f'{target}_shift',
        x='quantile_middle',
        title=feature0 + ' ' + target
    )
    fig.show()

In [ ]:
feature1 = 'O00533'
quantiles = [0, 0.05, 0.95, 1.0]

df_plot = []
for quantile_low, quantile_high in tqdm(zip(quantiles[:-1], quantiles[1:])):
    item = {
        'quantile_low': quantile_low,
        'quantile_high': quantile_high,
        'quantile_middle': (quantile_low + quantile_high) / 2
    }
    quantile_low_value = train_clinical_all[feature1].quantile(quantile_low)
    quantile_high_value = train_clinical_all[feature1].quantile(quantile_high)
    item['quantile_low_value'] = quantile_low_value
    item['quantile_high_value'] = quantile_high_value
    
    if quantile_high == 1:
        quantile_high_value += 0.00001
        
    train_clinical_all_filtered1 = train_clinical_all[
        (train_clinical_all[feature1] >= quantile_low_value)
        & (train_clinical_all[feature1] < quantile_high_value)
    ]
    for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
        item[f'{target}_shift'] = find_best_const(train_clinical_all_filtered1, target)
    df_plot.append(item)
    
df_plot = pd.DataFrame(df_plot)

In [ ]:
for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
    fig = px.line(
        df_plot,
        y=f'{target}_shift',
        x='quantile_middle',
        title=feature1 + ' ' + target
    )
    fig.show()

In [ ]:
feature2 = 'O15394'
quantiles = [0, 0.05, 0.95, 1.0]

df_plot = []
for quantile_low, quantile_high in tqdm(zip(quantiles[:-1], quantiles[1:])):
    item = {
        'quantile_low': quantile_low,
        'quantile_high': quantile_high,
        'quantile_middle': (quantile_low + quantile_high) / 2
    }
    quantile_low_value = train_clinical_all[feature2].quantile(quantile_low)
    quantile_high_value = train_clinical_all[feature2].quantile(quantile_high)
    item['quantile_low_value'] = quantile_low_value
    item['quantile_high_value'] = quantile_high_value
    
    if quantile_high == 1:
        quantile_high_value += 0.00001
        
    train_clinical_all_filtered2 = train_clinical_all[
        (train_clinical_all[feature2] >= quantile_low_value)
        & (train_clinical_all[feature2] < quantile_high_value)
    ]
    for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
        item[f'{target}_shift'] = find_best_const(train_clinical_all_filtered2, target)
    df_plot.append(item)
    
df_plot = pd.DataFrame(df_plot)

In [ ]:
for target in ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']:
    fig = px.line(
        df_plot,
        y=f'{target}_shift',
        x='quantile_middle',
        title=feature2 + ' ' + target
    )
    fig.show()

## Find shifts

In [ ]:
npx_groups = [
    {'quantile_low': 0.0, 'quantile_high': 0.05},
    {'quantile_low': 0.05, 'quantile_high': 0.95},
    {'quantile_low': 0.95, 'quantile_high': 1.0},
]
target_to_npx_groups_shift0 = defaultdict(list)

for target in ['updrs_1', 'updrs_2', 'updrs_3']:
    for npx_group in npx_groups:
        item = npx_group.copy()
        item['feature'] = feature0
        
        if item['quantile_low'] == 0:
            item['quantile_low_value'] = -np.inf
        else:
            item['quantile_low_value'] = train_clinical_all[feature0].quantile(item['quantile_low'])
            
        if item['quantile_high'] == 1:
            item['quantile_high_value'] = np.inf
        else: 
            item['quantile_high_value'] = train_clinical_all[feature0].quantile(item['quantile_high'])

        train_clinical_all_filtered0 = train_clinical_all[
            (train_clinical_all[feature0] >= item['quantile_low_value'])
            & (train_clinical_all[feature0] < item['quantile_high_value'])
        ]
        
        item['shift'] = find_best_const(train_clinical_all_filtered0, target)
        target_to_npx_groups_shift0[target].append(item)

In [ ]:
train_clinical_all.columns

In [ ]:
npx_groups = [
    {'quantile_low': 0.0, 'quantile_high': 0.05},
    {'quantile_low': 0.05, 'quantile_high': 0.95},
    {'quantile_low': 0.95, 'quantile_high': 1.0},
]
target_to_npx_groups_shift1 = defaultdict(list)

for target in ['updrs_1', 'updrs_2', 'updrs_3']:
    for npx_group in npx_groups:
        item = npx_group.copy()
        item['feature'] = feature1
        
        if item['quantile_low'] == 0:
            item['quantile_low_value'] = -np.inf
        else:
            item['quantile_low_value'] = train_clinical_all_filtered1[feature1].quantile(item['quantile_low'])
            
        if item['quantile_high'] == 1:
            item['quantile_high_value'] = np.inf
        else: 
            item['quantile_high_value'] = train_clinical_all_filtered1[feature1].quantile(item['quantile_high'])

        train_clinical_all_filtered1 = train_clinical_all_filtered1[
            (train_clinical_all_filtered1[feature1] >= item['quantile_low_value'])
            & (train_clinical_all_filtered1[feature1] < item['quantile_high_value'])
        ]
        
        item['shift'] = find_best_const(train_clinical_all_filtered1, target)
        target_to_npx_groups_shift1[target].append(item)

In [ ]:
npx_groups = [
    {'quantile_low': 0.0, 'quantile_high': 0.05},
    {'quantile_low': 0.05, 'quantile_high': 0.95},
    {'quantile_low': 0.95, 'quantile_high': 1.0},
]
target_to_npx_groups_shift2 = defaultdict(list)

for target in ['updrs_1', 'updrs_2', 'updrs_3']:
    for npx_group in npx_groups:
        item = npx_group.copy()
        item['feature'] = feature2
        
        if item['quantile_low'] == 0:
            item['quantile_low_value'] = -np.inf
        else:
            item['quantile_low_value'] = train_clinical_all[feature2].quantile(item['quantile_low'])
            
        if item['quantile_high'] == 1:
            item['quantile_high_value'] = np.inf
        else: 
            item['quantile_high_value'] = train_clinical_all[feature2].quantile(item['quantile_high'])

        train_clinical_all_filtered2 = train_clinical_all[
            (train_clinical_all[feature2] >= item['quantile_low_value'])
            & (train_clinical_all[feature2] < item['quantile_high_value'])
        ]
        
        item['shift'] = find_best_const(train_clinical_all_filtered2, target)
        target_to_npx_groups_shift2[target].append(item)

## Predictions

In [ ]:
amp_pd_peptide.make_env.func_dict['__called__'] = False
env = amp_pd_peptide.make_env()   # initialize the environment
iter_test = env.iter_test()    # an iterator which loops over the test files

proteins_features_all = pd.DataFrame()
# The API will deliver four dataframes in this specific order:
for test_clinical_data, test_peptides, test_proteins, sample_submission in iter_test:
    sample_submission['patient_id'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[0]))
    sample_submission['visit_month'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[1]))
    sample_submission['target_name'] = sample_submission['prediction_id'].map(lambda x: 'updrs_' + x.split('_')[3])
    sample_submission['plus_month'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[5]))
    sample_submission['pred_month'] = sample_submission['visit_month'] + sample_submission['plus_month']
    sample_submission['visit_id'] = sample_submission['patient_id'].astype(str) + '_' + sample_submission['visit_month'].astype(str)
    
    proteins_features = pd.pivot_table(test_proteins, values='NPX', index='visit_id', columns='UniProt', aggfunc='sum')
    proteins_features['visit_id'] = proteins_features.index
    proteins_features_all = pd.concat([proteins_features_all, proteins_features])
    proteins_features_all['patient_id'] = proteins_features_all.index.map(lambda x: int(x.split('_')[0]))
    proteins_features_all[proteins_features.columns] = proteins_features_all.groupby('patient_id')[proteins_features.columns].\
                                                                                                   fillna(method='ffill')
    proteins_features = proteins_features_all.groupby('patient_id', as_index=False).last()
    
    sample_submission = sample_submission.merge(
        proteins_features,
        on='patient_id',
        how='left'
    )

    for i in range(1, 5):
        target = f'updrs_{i}'
        mask_target = sample_submission['target_name'] == target
        sample_submission.loc[mask_target, 'rating'] = calculate_month_trend_predicitons(
            pred_month=sample_submission.loc[mask_target, 'pred_month'],
            trend=target_to_trend[target]
        )
        

        for item0, item1, item2 in zip(target_to_npx_groups_shift0[target], target_to_npx_groups_shift1[target], target_to_npx_groups_shift2[target]):
            # Extract features and values from item0 and item1
            feature0 = item0['feature']
            feature1 = item1['feature']
            feature2 = item2['feature']
            quantile_low_value0 = item0['quantile_low_value']
            quantile_low_value1 = item1['quantile_low_value']
            quantile_high_value0 = item0['quantile_high_value']
            quantile_high_value1 = item1['quantile_high_value']
            quantile_low_value2 = item2['quantile_low_value']
            quantile_high_value2 = item2['quantile_high_value']
            shift0 = item0['shift']
            shift1 = item1['shift']
            shift2 = item2['shift']
            mask_feature_range0 = mask_target & (
            ((sample_submission[feature0] >= quantile_low_value0) & (sample_submission[feature0] < quantile_high_value0)) |
            ((sample_submission[feature2] >= quantile_low_value2) & (sample_submission[feature2] < quantile_high_value2)) | ((sample_submission[feature1] >= quantile_low_value1) & (sample_submission[feature1] < quantile_high_value1)) | ((sample_submission[feature0] >= quantile_low_value0) & (sample_submission[feature0] < quantile_high_value0))
             )
            #mask_feature_range1=mask_target & )
           
            mask_feature_range = mask_target & (
                (sample_submission[feature0] >= quantile_low_value0) &
                (sample_submission[feature0] < quantile_high_value0)
            )
#             if target=='updrs_1' or target=="updrs_2":
#                 median_shift = np.mean([shift0, shift1, shift2])
#                 from scipy.stats import mode
#                 shifts = [shift0, shift1, shift2]
#                 mode_shift = mode(shifts).mode[0]
#                 # Calculate the median shift using the mode
#                 median_shift = np.median([mode_shift])
#                 sample_submission.loc[mask_feature_range0, 'rating'] += median_shift
#             else:
#                 #median_shift = np.median([shift0, shift1])
            sample_submission.loc[mask_feature_range0, 'rating'] += shift0
            # Update 'rating' column with the median shift for the current iteration
            
        sample_submission.loc[mask_target, 'rating'] = np.round(sample_submission.loc[mask_target, 'rating'])

    # call the env.predict for every iteration
    env.predict(sample_submission[['prediction_id', 'rating']])